# Premier vrai LLM NeuroDSL — TinyShakespeare, char-level

Phase 1 du plan "vrai test sur un vrai LLM" : jusqu'ici, tout l'entraînement de ce dépôt (hors
MNIST) utilisait des IDs de tokens générés aléatoirement, jamais de vrai texte ni de vrai
tokenizer. Ce notebook entraîne un petit modèle de langage caractère-par-caractère sur du texte
réel (TinyShakespeare, domaine public, ~1.1 Mo, standard nanoGPT/char-rnn), avec un vrai
tokenizer (caractères uniques du corpus), un vrai split train/validation, et génère de vrais
échantillons de texte avant/pendant/après l'entraînement.

Réutilise entièrement l'infrastructure déjà testée : `Embedding`, `LlamaModel`, `Linear`,
`:cross_entropy`, le patron d'entraînement AdamW pas-à-pas (`src/synthetic_circuits.jl`) --
aucune modification de `src/` ni de `Project.toml` (`Downloads` est un stdlib Julia).

Une version PyTorch miroir stricte (même architecture, mêmes données, mêmes hyperparamètres)
vit dans `notebook/real_llm_py.py`, pour comparaison et vérification croisée de correction.

In [1]:
using NeuroDSL, Random, Statistics, Printf, StatsPlots

dev = NeuroDSL.Backend.CUDADevice()
ns = :real_llm
println("Device: ", dev)

Device: NeuroDSL.Backend.CUDADevice()


## 1. Corpus : TinyShakespeare (téléchargé une seule fois, hors ligne ensuite)

In [2]:
using Downloads

const CORPUS_URL  = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
const CORPUS_PATH = joinpath(@__DIR__, "data", "tinyshakespeare", "input.txt")

function load_corpus(path::String, url::String)
    if !isfile(path)
        mkpath(dirname(path))
        Downloads.download(url, path)
    end
    text = read(path, String)
    println("Corpus chargé : ", length(text), " caractères depuis ", path)
    return text
end

text = load_corpus(CORPUS_PATH, CORPUS_URL)
println(first(text, 200))

Corpus chargé : 1115394 caractères depuis C:\Users\Nevermind\Desktop\NeuroDSL\notebook\data\tinyshakespeare\input.txt
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


## 2. Tokenizer caractère (vrai texte -> vrais IDs, pas de BPE, zéro dépendance nouvelle)

In [3]:
function build_char_tokenizer(text::String)
    chars = sort(unique(collect(text)))
    stoi = Dict(c => i for (i, c) in enumerate(chars))
    return chars, stoi
end

encode(text::AbstractString, stoi::Dict{Char,Int}) = [stoi[c] for c in text]
decode(ids::AbstractVector{<:Integer}, chars::Vector{Char}) = String(chars[ids])

chars, stoi = build_char_tokenizer(text)
vocab_size = length(chars)
println("vocab_size = ", vocab_size)
println("10 premiers caractères (triés) = ", chars[1:10])
println("(à comparer avec les 10 premiers caractères imprimés par real_llm_py.py -- doivent être identiques)")

vocab_size = 65
10 premiers caractères (triés) = ['\n', ' ', '!', '$', '&', '\'', ',', '-', '.', '3']
(à comparer avec les 10 premiers caractères imprimés par real_llm_py.py -- doivent être identiques)


## 3. Split train/validation (90/10 par position -- le val est la FIN du corpus, jamais vue à l'entraînement)

In [4]:
data = encode(text, stoi)
n_total = length(data)
n_train = floor(Int, 0.9 * n_total)
train_ids = data[1:n_train]
val_ids   = data[n_train+1:end]
println("train: ", length(train_ids), " caractères  |  val: ", length(val_ids), " caractères")

train: 1003854 caractères  |  val: 111540 caractères


## 4. Construction du graphe (copie directe de `build_induction_graph`, généralisée à un vrai vocabulaire/contexte)

In [5]:
function build_char_lm_graph(dev, ns::Symbol; vocab_size::Int, dim::Int, n_heads::Int,
                              hidden_dim::Int, n_layers::Int, block_size::Int)
    g = NeuroDSL.NeuroGraph(namespace=ns, device=dev)
    NeuroDSL.set!(g, :token_ids, ones(Int, block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    tok_emb = NeuroDSL.Embedding(vocab_size, dim)(g, :token_ids, :tok; namespace=ns)
    pos_emb = NeuroDSL.Embedding(block_size, dim)(g, :pos_ids, :pos; namespace=ns)
    x = :embed_sum
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(x, [tok_emb, pos_emb], :add; namespace=ns))
    out = NeuroDSL.LlamaModel(n_layers, dim, n_heads, hidden_dim)(g, x; namespace=ns)
    logits = NeuroDSL.Linear(dim, vocab_size)(g, out, :lm_head; namespace=ns)
    NeuroDSL.set!(g, :labels, ones(Int, block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [logits, :labels], :cross_entropy; namespace=ns))
    return g, logits
end

# ── Hyperparamètres (voir plan : dimensionnés pour rester bien sous le max GPU
# déjà confirmé cette session -- dim=1024 en forward+backward réel) ──────────
block_size = 256
dim        = 256
n_heads    = 4
hidden_dim = 512
n_layers   = 4

g, logits_sym = build_char_lm_graph(dev, ns; vocab_size=vocab_size, dim=dim, n_heads=n_heads,
                                     hidden_dim=hidden_dim, n_layers=n_layers, block_size=block_size)
ps = NeuroDSL.params(g; namespace=ns)
n_scalars = sum(length(p.value) for p in ps)
println("Graphe : ", length(g.nodes[ns]), " nœuds, ", length(ps), " tenseurs de paramètres, ",
        n_scalars, " scalaires (~", round(n_scalars/1e6, digits=2), "M)")

Graphe : 212 nœuds, 40 tenseurs de paramètres, 2722369 scalaires (~2.72M)


## 5. Échantillonnage de fenêtres réelles + perte de validation + génération autorégressive

In [6]:
function sample_window(rng, ids::Vector{Int}, block_size::Int)
    i = rand(rng, 1:(length(ids) - block_size))
    tokens = ids[i:i+block_size-1]
    labels = ids[i+1:i+block_size]
    return tokens, labels
end

# 64 fenêtres FIXES également espacées dans le split val -- déterministe,
# comparable entre checkpoints. Jamais de backward_graph! ici (pas de fuite
# du val dans les gradients).
function val_loss(g::NeuroDSL.NeuroGraph, ns::Symbol; val_ids::Vector{Int}, block_size::Int, n_windows::Int=64)
    max_start = length(val_ids) - block_size
    starts = round.(Int, range(1, max_start, length=n_windows))
    total = 0.0
    for i in starts
        tokens = val_ids[i:i+block_size-1]
        labels = val_ids[i+1:i+block_size]
        NeuroDSL.set!(g, :token_ids, tokens; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :labels, labels; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.invalidate_all!(g; namespace=ns)
        loss_val = NeuroDSL.demand!(g, :loss; namespace=ns)
        total += Float64(sum(Array(loss_val)))
    end
    return total / n_windows
end

# Génération autorégressive -- échantillonnage AVEC TEMPÉRATURE (pas argmax) :
# l'argmax sur un char-LM dégénère quasi systématiquement en boucles
# répétitives ("the the the..."), donnant une fausse impression d'échec alors
# que la distribution apprise est bonne. C'est ce que nanoGPT/char-rnn font
# pour leurs démos.
function generate_text(g::NeuroDSL.NeuroGraph, logits_sym::Symbol, ns::Symbol,
                        stoi::Dict{Char,Int}, chars::Vector{Char};
                        seed_text::String="\n", n_chars::Int=300, temperature::Float32=0.8f0,
                        block_size::Int, rng=MersenneTwister(777), use_argmax::Bool=false)
    ctx = encode(seed_text, stoi)
    generated = Char[]
    for _ in 1:n_chars
        window = length(ctx) > block_size ? ctx[end-block_size+1:end] : ctx
        t = length(window)
        NeuroDSL.set!(g, :token_ids, window; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :pos_ids, collect(1:t); atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.invalidate_all!(g; namespace=ns)
        row = Array(NeuroDSL.demand!(g, logits_sym; namespace=ns))[end, :]
        local next_id
        if use_argmax
            next_id = argmax(row)
        else
            p = exp.((row .- maximum(row)) ./ temperature)
            p ./= sum(p)
            r = rand(rng)
            cum = 0.0f0
            next_id = length(p)
            for (idx, pi) in enumerate(p)
                cum += pi
                if r <= cum
                    next_id = idx
                    break
                end
            end
        end
        push!(ctx, next_id)
        push!(generated, chars[next_id])
    end
    return String(generated)
end

generate_text (generic function with 1 method)

## 6. Vérification de sanité : perte initiale ≈ ln(vocab_size)

In [7]:
rng_check = MersenneTwister(1)
tokens0, labels0 = sample_window(rng_check, train_ids, block_size)
NeuroDSL.set!(g, :token_ids, tokens0; atom_type=NeuroDSL.Datom, namespace=ns)
NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
NeuroDSL.set!(g, :labels, labels0; atom_type=NeuroDSL.Datom, namespace=ns)
NeuroDSL.invalidate_all!(g; namespace=ns)
loss0 = Float64(sum(Array(NeuroDSL.demand!(g, :loss; namespace=ns))))
@printf("Perte initiale mesurée : %.4f   (attendu ln(%d) = %.4f)\n", loss0, vocab_size, log(vocab_size))
@assert abs(loss0 - log(vocab_size)) < 0.5 "Perte initiale trop loin de ln(vocab_size) -- vérifier le câblage avant d'entraîner"

println("\n--- Échantillon AVANT tout entraînement (poids aléatoires) ---")
sample_before = generate_text(g, logits_sym, ns, stoi, chars; block_size=block_size, rng=MersenneTwister(777))
println(sample_before)

Perte initiale mesurée : 4.2144   (attendu ln(65) = 4.1744)

--- Échantillon AVANT tout entraînement (poids aléatoires) ---
;Bsqoj
MWMdi$EWxfRgtDR!N,ynJEM uKJ!xf&!MqHI!Z$!'
oWftk-innbYBA tjZ,xxJXjRfYxNm?a-nrW
ZoOTM
xci3a
mpw3NUdktlSp
$yx
$nNNDFjsK3sM3YLn'ZT,nxkhQXN.dwBWaAPopcjsMtaUezi
i&W Ff
 HgWT
.k'NIqs$wgI.nQ,yKVq:WupsDCY-R'uIosCZKdU
vZy, HdywCDLmWvWp;vw,3woxRl.zR!tmk!,d$zL
AKzYAYHFHV?MkmT.pkm kRbeS-YB j'kg&oK3Ibhoy$u


## 7. Sonde de timing (mesurée, pas supposée)

Sonde exécutée séparément avant ce notebook (script scratch, même architecture/mêmes
hyperparamètres exacts) : **26.36 ms/pas** (médiane des pas 11-50, GPU NVIDIA RTX A5500).
À ce rythme, 10 000 pas prendraient ~4.4 minutes -- largement sous le budget de 45 minutes
prévu par le plan. Décision : **`n_steps = 20 000`** (~8.8 minutes estimées), soit
`20000 * 256 / 1 003 854 ≈ 5.1` passages équivalents sur tout le corpus d'entraînement --
un compromis entre convergence réelle et temps d'exécution, avec de la marge sous le budget
initial. Le chronométrage du run complet ci-dessous (§8) est le chiffre qui sera publié, pas
cette sonde préliminaire.

## 8. Entraînement complet

In [8]:
function train_char_lm!(g::NeuroDSL.NeuroGraph, ns::Symbol, logits_sym::Symbol,
                         stoi::Dict{Char,Int}, chars::Vector{Char};
                         train_ids::Vector{Int}, val_ids::Vector{Int}, block_size::Int,
                         n_steps::Int, lr::Float32=1f-3, b1::Float32=0.9f0, b2::Float32=0.999f0,
                         eps_v::Float32=1f-8, clip::Float32=1f0, wd::Float32=0f0, seed::Int=123,
                         val_every::Int=500, sample_steps=(500, 2500, 10000))
    dev = g.device
    ps = NeuroDSL.params(g; namespace=ns)
    m1s = [NeuroDSL.Backend.zeros32(dev, size(p.value)...) for p in ps]
    m2s = [NeuroDSL.Backend.zeros32(dev, size(p.value)...) for p in ps]
    rng = MersenneTwister(seed)
    train_losses = Float64[]
    val_history = Tuple{Int,Float64}[]

    t_start = time()
    for t in 1:n_steps
        tokens, labels = sample_window(rng, train_ids, block_size)
        NeuroDSL.set!(g, :token_ids, tokens; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :labels, labels; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.invalidate_all!(g; namespace=ns)
        loss_val = NeuroDSL.demand!(g, :loss; namespace=ns)
        push!(train_losses, Float64(sum(Array(loss_val))))
        NeuroDSL.backward_graph!(g, :loss; namespace=ns)
        for (i, p) in enumerate(ps)
            NeuroDSL.adamw_step!(dev, p.value, p.gradient, m1s[i], m2s[i], lr, b1, b2, eps_v, t, clip, wd)
        end
        NeuroDSL.invalidate_all!(g; namespace=ns)

        if t % val_every == 0 || t == 1
            vl = val_loss(g, ns; val_ids=val_ids, block_size=block_size)
            push!(val_history, (t, vl))
            @printf("step %6d | train %.4f | val %.4f | ppl %.2f | bits/char %.3f\n",
                    t, train_losses[end], vl, exp(vl), vl/log(2))
        end
        if t in sample_steps
            s = generate_text(g, logits_sym, ns, stoi, chars; block_size=block_size, rng=MersenneTwister(777))
            println("\n--- Échantillon @ pas $t ---\n", s, "\n")
        end
    end
    elapsed = time() - t_start

    return (; train_losses, val_history, elapsed)
end

n_steps = 20_000
result = train_char_lm!(g, ns, logits_sym, stoi, chars;
                         train_ids=train_ids, val_ids=val_ids, block_size=block_size,
                         n_steps=n_steps, lr=1f-3)
@printf("\nEntraînement terminé : %d pas en %.1f s (%.2f ms/pas moyen)\n",
        n_steps, result.elapsed, 1000*result.elapsed/n_steps)
println("(Mesure du pic VRAM : voir le script séparé real_llm_vram_probe.jl -- doit tourner isolé,")
println(" voir son en-tête pour la raison exacte.)")

step      1 | train 4.2059 | val 3.8415 | ppl 46.60 | bits/char 5.542
step    500 | train 2.5989 | val 2.5539 | ppl 12.86 | bits/char 3.684

--- Échantillon @ pas 500 ---
CETisT le sid owinit m mayol l we ay.

Bul ar ad
Therilloure d use womms.
Hr t patot pr me
cis o tit leisther

Xr athe foumer be t or tour, l steLO:
Sours, rasus
MEN fi LENTI bo l to win thay nod wite o mauloule ra who anestameraver wo crmy t y ausicar y MENur on:
F:
PUS:
QU, moun n' t th thanenow w

step   1000 | train 2.3826 | val 2.5468 | ppl 12.77 | bits/char 3.674
step   1500 | train 2.5060 | val 2.5093 | ppl 12.30 | bits/char 3.620
step   2000 | train 2.4971 | val 2.5295 | ppl 12.55 | bits/char 3.649
step   2500 | train 2.3336 | val 2.3472 | ppl 10.46 | bits/char 3.386

--- Échantillon @ pas 2500 ---
CETSAT I fot eethedoin hawin.

The yor himacid a therolllle.


The wordd thyat patou ou, h witel
Wit hellther

pu at he tol wharat pidowit pe prekiinseert withy,
The fo anghe buchaticte bur yout nout clan whit orfia

## 9. Échantillon final + comparaison argmax, et courbes

In [9]:
println("--- Échantillon APRÈS entraînement (température=0.8) ---")
sample_after = generate_text(g, logits_sym, ns, stoi, chars; block_size=block_size,
                              rng=MersenneTwister(777), n_chars=400)
println(sample_after)

println("\n--- Échantillon APRÈS entraînement (argmax, pour comparaison documentée) ---")
sample_after_argmax = generate_text(g, logits_sym, ns, stoi, chars; block_size=block_size,
                                     rng=MersenneTwister(777), n_chars=200, use_argmax=true)
println(sample_after_argmax)

final_val = result.val_history[end][2]
@printf("\nVal loss finale : %.4f nats/char  (perplexité %.2f, %.3f bits/char)\n",
        final_val, exp(final_val), final_val/log(2))

--- Échantillon APRÈS entraînement (température=0.8) ---
Bestrible the love's man worged:
I will the pain the sentence are as her spressenger
To prayer the warmers, takes and fllow up,
My prayer of for the state the woar of tears.

DUCHESS OF YORK:
No, with see were she I not know me.

KING HENRY VII:
Will test hevell fllow the stay son of the are
To trave within the her it a next true from off the tempent-bood you behalf order in
though lip us some to 

--- Échantillon APRÈS entraînement (argmax, pour comparaison documentée) ---
The the contraction the contraction.

KING RICHARD III:
Not stand the great the greater the grace,
The greater the stand the great the greaters
To the greater the time the contract the time
To the pro

Val loss finale : 1.6937 nats/char  (perplexité 5.44, 2.443 bits/char)


In [10]:
plot(1:length(result.train_losses), result.train_losses, label="train loss (par pas)",
     alpha=0.4, color=:steelblue, xlabel="pas", ylabel="perte (nats/char)",
     title="NeuroDSL -- char-LM sur TinyShakespeare")
val_x = [v[1] for v in result.val_history]
val_y = [v[2] for v in result.val_history]
plot!(val_x, val_y, label="val loss", color=:orange, lw=2, marker=:circle, markersize=3)
hline!([log(vocab_size)], label="ln(vocab_size) -- niveau aléatoire", linestyle=:dash, color=:gray)
savefig(joinpath(@__DIR__, "..", "figures", "real_llm_loss.pdf"))
println("Figure sauvegardée -> figures/real_llm_loss.pdf")

Figure sauvegardée -> figures/real_llm_loss.pdf


## 10. Sauvegarde des résultats pour la comparaison avec `real_llm_py.py`

Écrit les métriques clés dans un petit fichier JSON -- lu par le script de comparaison finale
(section 11), pas de dépendance à l'ordre d'exécution des deux notebooks/scripts.

In [ ]:
using JSON

results = Dict(
    "vocab_size" => vocab_size,
    "chars" => String(chars),
    "n_params" => n_scalars,
    "n_steps" => n_steps,
    "elapsed_s" => result.elapsed,
    "ms_per_step" => 1000*result.elapsed/n_steps,
    "initial_loss" => loss0,
    "final_train_loss" => result.train_losses[end],
    "final_val_loss" => final_val,
    "peak_vram_mb" => 91.78,  # mesuré séparément par real_llm_vram_probe.jl (isolé) -- pic retenu = train_step, après GradPool (src/grad_pool.jl : pool de gradients à propriétaire unique dans backward_graph!) + pooling de dA pour :matmul ; val_window=81.70, gen_token=81.80 (PyTorch=73.6, ratio ~1.11x)
    "val_history" => result.val_history,
    "sample_before" => sample_before,
    "sample_after" => sample_after,
)
open(joinpath(@__DIR__, "real_llm_neurodsl_results.json"), "w") do io
    JSON.print(io, results)
end
println("Résultats écrits -> notebook/real_llm_neurodsl_results.json")